# Projeto 01 — Agente Inteligente de Otimização de Rotas de Distribuição Alimentar Local
## Unidade Curricular de Inteligência Artificial — IPCA (2026/2027)
### Entrega Preliminar (Fase 1): Modelação, Análise Exploratória e Validador de Rotas

---

### Identificação do Grupo (Grupo 04 — `IA26_P01_G04`)
* **Gonçalo Correia** — Nº 22840
* **Bruno Dantas** — Nº 27958
* **Ricardo Ferreira** — Nº 25426
* **Nuno Gajo** — Nº 23002

**Repositório Oficial:** [https://github.com/BrunoDantas1/IA26_P01_G04.git](https://github.com/BrunoDantas1/IA26_P01_G04.git)

---

## 1. Introdução e Contextualização do Problema

### 1.1. Enquadramento
A transição para cadeias de abastecimento alimentar curtas e sustentáveis coloca desafios logísticos complexos. Pequenos produtores agrícolas locais operam com margens temporais reduzidas de frescura dos produtos, enquanto os consumidores finais e mercados municipais exigem entregas em horários de conveniência bem definidos.

Neste projeto desenvolvemos um **Agente Inteligente de Planeamento de Rotas** para distribuição alimentar local na região do distrito de **Braga** e concelho da **Póvoa de Varzim**. O problema enquadra-se formalmente na classe dos **Pickup-and-Delivery Problems with Time Windows and Capacity Constraints (PDPTW)**, que combinam restrições espaciais, de precedência, temporais e de capacidade.

### 1.2. Atores e Componentes do Sistema
1. **Produtores Locais (*Producers*)**: Locais de origem onde os produtos agrícolas são colhidos/embalados e disponibilizados dentro da janela regulamentar $[\text{availability\_start}, \text{availability\_end}]$.
2. **Centros de Recolha / Armazéns (*CollectionCenters*)**: Plataformas logísticas centrais onde os bens podem ser agregados e que servem de depósito base permanente da frota de veículos.
3. **Locais de Entrega (*DeliveryLocations / Customers*)**: Clientes residenciais ou mercados municipais com janelas de entrega $[\text{delivery\_start}, \text{delivery\_end}]$.
4. **Frota de Veículos (*Vehicles*)**: Carrinhas comerciais com capacidades máximas específicas (1444.7 kg e 1009.8 kg) que realizam as viagens.
5. **Matriz de Distâncias e Tempos (*DistanceMatrix*)**: Baseada na rede de paragens unificadas de `locations.csv`, fornecendo distâncias em km (distância ortodrómica corrigida com fator rodoviário de 1.3) e tempos de viagem em minutos calculados a uma velocidade de 45 km/h.

### 1.3. Objetivos desta Entrega Preliminar
* **Exploratory Data Analysis (EDA)**: Carregamento dos dados CSV, verificação exaustiva de integridade referencial, perfil da procura e consistência das janelas temporais.
* **Modelação Orientada a Objetos**: Implementação do módulo `model.py` com representação explícita de `Request`, `Vehicle`, `State`, `Problem`, função de custo (`cost_function`) e validador de viabilidade (`route_feasibility_checker`).
* **Validação por Rotas Manuais (*Hand-Written Routes*)**: Demonstração empírica do detetor de viabilidade através de cenários válidos e cenários inválidos (precedência, janelas temporais e capacidade).


In [1]:
# Configuração do ambiente e importação do módulo de modelação
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Importar classes do modelo desenvolvido
from model import Problem, Request, Vehicle, State

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120

print("Módulo model.py importado com sucesso!")


Módulo model.py importado com sucesso!


## 2. Exploratory Data Analysis (EDA) — Síntese Executiva

Realizámos previamente uma análise exaustiva no notebook complementar `01_eda_tiny.ipynb` (e cópia `EDA.ipynb`). Apresentamos de seguida a síntese dos principais resultados que sustentam as decisões de desenho do nosso modelo de IA:

### 2.1. Integridade Relacional e Mapeamento Unificado de Paragens
* Todas as 11 tabelas do dataset possuem chaves primárias únicas e **zero chaves estrangeiras órfãs**.
* **Identificação Canónica de Paragens**: A tabela `locations.csv` estabelece uma chave unificada `location_id` entre 1 e 12:
  * `location_id` 1 a 5: Produtores agrícolas (`location_type == 'producer'`);
  * `location_id` 6: Centro de Recolha 1 (`location_type == 'collection_center'`);
  * `location_id` 7 a 12: Locais de Entrega (`location_type == 'delivery_location'`).
  * O nosso módulo `model.py` resolve automaticamente qualquer ambiguidade, mapeando `(location_type, ref_id) -> location_id`.

### 2.2. Capacidade da Frota vs. Procura das Encomendas
* **Veículo 1 (ZG-06-DP)**: Capacidade de **1444.7 kg**.
* **Veículo 2 (TY-74-YA)**: Capacidade de **1009.8 kg**.
* **Centro de Recolha**: Capacidade de **3232.9 kg**.
* **Procura Global do Dataset**: 23 artigos totalizando **248.96 kg** (peso médio por artigo de 10.82 kg e máximo de 24.60 kg).
* A capacidade global da frota é muito superior à procura total, permitindo consolidar múltiplos pedidos na mesma carrinha. As restrições de capacidade atuam como verificador contínuo em cada ponto da rota.

### 2.3. Coerência Temporal do Dataset Sintético
* Na análise de consistência temporal apurámos que 12 artigos possuem `order_datetime` posterior ao fecho da janela de recolha (`availability_end`), constituindo um artefacto comum em geradores sintéticos de dados.
* Restam **11 artigos perfeitamente coerentes e viáveis**, dos quais se destacam os pedidos do dia **19 de Outubro de 2026** (ex: artigos 7, 10 e 22) e do dia **10 de Outubro de 2026** (artigo 5), que formam o nosso cenário de referência.


In [2]:
# Carregamento e verificação do Problema através do model.py
data_path = "data_tiny" if os.path.exists("data_tiny/producers.csv") else ("data" if os.path.exists("data/producers.csv") else "data_tiny/data_tiny")
problem = Problem(data_folder=data_path)

print(f"Instância carregada com sucesso!")
print(f"Total de Veículos: {len(problem.vehicles)}")
for vid, v in problem.vehicles.items():
    print(f"  -> Veículo {vid}: Matrícula {v.plate} | Centro Loc {v.center_location_id} | Capacidade {v.capacity_kg:.1f} kg")

# Filtrar artigos temporalmente consistentes para o benchmark
valid_requests = [r for r in problem.requests if r.ready_time <= r.availability_end]
problem.requests = valid_requests
problem.requests_by_id = {r.item_id: r for r in problem.requests}

print(f"Total de Pedidos Coerentes e Elegíveis: {len(problem.requests)}")


Instância carregada com sucesso!
Total de Veículos: 2
  -> Veículo 1: Matrícula ZG-06-DP | Centro Loc 6 | Capacidade 1444.7 kg
  -> Veículo 2: Matrícula TY-74-YA | Centro Loc 6 | Capacidade 1009.8 kg
Total de Pedidos Coerentes e Elegíveis: 11


## 3. Modelação Orientada a Objetos e Testes de Viabilidade

### 3.1. Formulação do Problema
O problema de planeamento de rotas foi formulado com base nas classes centrais implementadas em `model.py`:
* `Request`: Modela cada artigo de encomenda com nó de recolha (`pickup_location_id`), nó de entrega (`delivery_location_id`), peso em kg e janelas temporais $[ready\_time, availability\_end]$ e $[delivery\_start, delivery\_end]$.
* `Vehicle`: Caracteriza o veículo da frota, identificando o seu armazém base de partida/regresso e capacidade de carga máxima.
* `State`: Representa o estado do agente de pesquisa num determinado instante: localização atual, momento cronológico, carga embarcada, conjunto de artigos recolhidos, artigos em trânsito e artigos entregues.
* `Problem`: Motor central que indexa as matrizes de distância e tempo, gera ações admissíveis e computa a função de custo.

### 3.2. Função de Custo (`cost_function`)
Para uma rota $\sigma = (s_0, s_1, \dots, s_k)$, a função de custo calcula:
1. **Distância Total Percorrida ($km$)**: Soma das distâncias ortodrómicas com fator rodoviário de 1.3:
$$D(\sigma) = \sum_{i=0}^{k-1} dist(s_i, s_{i+1})$$
2. **Duração Total ($minutos$)**: Soma do tempo efetivo de viagem a 45 km/h mais os tempos de espera forçada sempre que o veículo chega a um produtor ou cliente antes da abertura da janela temporal:
$$T(\sigma) = \sum_{i=0}^{k-1} travel\_time(s_i, s_{i+1}) + \sum_{i=1}^k wait\_time(s_i)$$

### 3.3. Validador Rigoroso de Viabilidade (`route_feasibility_checker`)
O validador verifica exaustivamente:
1. **Origem e Retorno no Depósito**: A rota inicia e conclui obrigatoriamente no `CollectionCenter` do veículo.
2. **Precedência Estrita**: Nenhum artigo pode ser entregue sem ter sido previamente recolhido no seu produtor.
3. **Respeito Absoluto pela Capacidade**: A carga atual nunca pode exceder `capacity_kg` do veículo em nenhum momento da rota.
4. **Janelas Temporais**: O veículo não pode chegar a um local após o fecho da respetiva janela ($arrival \le deadline$). Se chegar antes da abertura, aguarda legitimamente pelo início do serviço.


### 3.4. Demonstração com Rotas Escritas à Mão (*Hand-Written Routes*)
Para comprovar que o `route_feasibility_checker` e a `cost_function` detetam qualquer anomalia e calculam os custos reais, executamos testes com **2 rotas perfeitamente válidas** e **3 rotas inválidas** (violando precedência, tempo e capacidade).


In [3]:
# -------------------------------------------------------------
# TESTE 1: ROTA VÁLIDA 1 (1 Artigo: Item 5 no dia 10 de Outubro)
# -------------------------------------------------------------
start_t1 = pd.Timestamp("2026-10-10 12:00:00")
rota_valida_1 = [
    ("start", 6),       # Parte do armazém base (Loc 6)
    ("pickup", 5),      # Recolhe Item 5 no produtor Pomar Carvalho (Loc 4)
    ("delivery", 5),    # Entrega Item 5 no cliente Casa 4 (Loc 10)
    ("return", 6)       # Regressa ao armazém base (Loc 6)
]

viavel1, msg1, det1 = problem.route_feasibility_checker(rota_valida_1, vehicle_id=1, start_time=start_t1)
print(f"=== TESTE 1: ROTA VÁLIDA 1 ===")
print(f"Viável? {viavel1} | Mensagem: {msg1}")
if viavel1:
    print(f"Distância Total: {det1['total_distance_km']} km")
    print(f"Tempo Total: {det1['total_time_min']} min (Viagem: {det1['total_travel_time_min']} min | Espera: {det1['total_wait_time_min']} min)")
    print(f"Itens Entregues com Sucesso: {det1['delivered_items_count']}")


=== TESTE 1: ROTA VÁLIDA 1 ===
Viável? True | Mensagem: Rota perfeitamente viável! Cumpre todas as restrições.
Distância Total: 164.51 km
Tempo Total: 241.7 min (Viagem: 219.4 min | Espera: 22.3 min)
Itens Entregues com Sucesso: 1


In [4]:
# -------------------------------------------------------------------------
# TESTE 2: ROTA VÁLIDA 2 (2 Artigos no mesmo dia: Itens 10 e 7 em 19 Outubro)
# -------------------------------------------------------------------------
start_t2 = pd.Timestamp("2026-10-19 07:30:00")
rota_valida_2 = [
    ("start", 6),       # Depósito base (Loc 6)
    ("pickup", 10),     # Recolhe Item 10 no produtor (Loc 4)
    ("delivery", 10),   # Entrega Item 10 no cliente (Loc 8)
    ("pickup", 7),      # Recolhe Item 7 no produtor (Loc 4)
    ("delivery", 7),    # Entrega Item 7 no cliente (Loc 10)
    ("return", 6)       # Regresso ao armazém base (Loc 6)
]

viavel2, msg2, det2 = problem.route_feasibility_checker(rota_valida_2, vehicle_id=1, start_time=start_t2)
print(f"=== TESTE 2: ROTA VÁLIDA 2 ===")
print(f"Viável? {viavel2} | Mensagem: {msg2}")
if viavel2:
    print(f"Distância Total: {det2['total_distance_km']} km")
    print(f"Tempo Total: {det2['total_time_min']} min (Viagem: {det2['total_travel_time_min']} min | Espera: {det2['total_wait_time_min']} min)")


=== TESTE 2: ROTA VÁLIDA 2 ===
Viável? True | Mensagem: Rota perfeitamente viável! Cumpre todas as restrições.
Distância Total: 263.23 km
Tempo Total: 351.0 min (Viagem: 351.0 min | Espera: 0.0 min)


In [5]:
# -------------------------------------------------------------------------
# TESTE 3: ROTA INVÁLIDA 1 — Violação de Precedência (Entrega antes da Recolha)
# -------------------------------------------------------------------------
rota_invalida_precedencia = [
    ("start", 6),
    ("delivery", 5),    # Tentativa ilícita de entregar o Item 5 antes de o recolher!
    ("pickup", 5),
    ("return", 6)
]

viavel3, msg3, _ = problem.route_feasibility_checker(rota_invalida_precedencia, vehicle_id=1, start_time=start_t1)
print(f"=== TESTE 3: INVÁLIDA POR PRECEDÊNCIA ===")
print(f"Viável? {viavel3} -> {msg3}")


=== TESTE 3: INVÁLIDA POR PRECEDÊNCIA ===
Viável? False -> Violação de Precedência no passo 1: Tentativa de entrega do Item 5 sem ter sido previamente recolhido!


In [6]:
# -------------------------------------------------------------------------
# TESTE 4: ROTA INVÁLIDA 2 — Violação de Janela Temporal (Chegada Tardia)
# -------------------------------------------------------------------------
# Se o veículo iniciar às 16:00, chegará ao produtor após as 15:00 (fecho da disponibilidade)
start_t_tarde = pd.Timestamp("2026-10-10 16:00:00")
rota_invalida_tempo = [
    ("start", 6),
    ("pickup", 5),
    ("delivery", 5),
    ("return", 6)
]

viavel4, msg4, _ = problem.route_feasibility_checker(rota_invalida_tempo, vehicle_id=1, start_time=start_t_tarde)
print(f"=== TESTE 4: INVÁLIDA POR JANELA TEMPORAL ===")
print(f"Viável? {viavel4} -> {msg4}")


=== TESTE 4: INVÁLIDA POR JANELA TEMPORAL ===
Viável? False -> Violação de Janela Temporal de Recolha no passo 1 (Item 5): Chegada às 2026-10-10 17:26:42, após o fecho da janela (2026-10-10 15:00:00) com atraso de 146.7 minutos.


In [7]:
# -------------------------------------------------------------------------
# TESTE 5: ROTA INVÁLIDA 3 — Violação de Capacidade Máxima do Veículo
# -------------------------------------------------------------------------
# Criamos um pedido simulado com 2000 kg (excedendo a capacidade do Veículo 1 de 1444.7 kg)
class SyntheticHeavyRequest:
    def __init__(self, base_req, weight):
        for k, v in base_req.__dict__.items():
            setattr(self, k, v)
        self.weight_kg = weight

problem.requests_by_id[999] = SyntheticHeavyRequest(problem.requests_by_id[5], 2000.0)

rota_invalida_cap = [
    ("start", 6),
    ("pickup", 999),
    ("delivery", 999),
    ("return", 6)
]

viavel5, msg5, _ = problem.route_feasibility_checker(rota_invalida_cap, vehicle_id=1, start_time=start_t1)
print(f"=== TESTE 5: INVÁLIDA POR EXCESSO DE CAPACIDADE ===")
print(f"Viável? {viavel5} -> {msg5}")


=== TESTE 5: INVÁLIDA POR EXCESSO DE CAPACIDADE ===
Viável? False -> Violação de Capacidade no passo 1: Carga total atingiria 2000.00 kg, excedendo o limite de 1444.70 kg do veículo em 555.30 kg.


## 4. Problem Solving (Algoritmos de IA)

> **Nota:** Esta secção encontra-se reservada para a **Entrega Final (Fase 2)** do projeto, onde serão detalhados e comparados:
> * **Informed Search**: Greedy Best-First Search e A* Search com 2 heurísticas admissíveis ($h_1$ baseada na soma dos percursos diretos e $h_2$ baseada no gargalo de maior desvio).
> * **Local Search**: Hill-Climbing com operador 2-opt a partir da rota inicial gerada pelo Greedy (apresentando a curva de Custo vs. Iteração).
> * **Constraint Satisfaction Problem (CSP)**: Formulação com variáveis, domínios, restrições e resolução via Backtracking com podas/backtracks.
> * **Coordenação da Frota de 2 Veículos**: Apresentação das rotas finais detalhadas para os Veículos 1 e 2.


## 5. Result Analysis & Conclusion

> **Nota:** Esta secção será desenvolvida na **Entrega Final (Fase 2)** com a inclusão de:
> * Tabela comparativa e gráficos consolidados com métricas de distância, tempo, nós expandidos e backtracks.
> * Análise crítica das heurísticas e algoritmos.
> * Limitações do modelo, propostas de melhoria futura e conclusões finais da equipa.
